In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


%store -r df_eda_v1

In [ ]:
list(df_eda_v1.columns)

['market_id',
 'market_slug',
 'crypto',
 'timeframe',
 'resolution_status',
 'winning_outcome_label',
 'resolution',
 'opens_at',
 'close_at',
 'resolved_at',
 'start_ts',
 'end_ts',
 'window_start_ts',
 'prediction_ts',
 'price_timestamp',
 'price_age_seconds',
 'effective_seconds_to_close',
 'up_price',
 'down_price',
 'price_second_trade_count',
 'price_second_volume_usdc',
 'window_trade_count',
 'window_volume_usdc',
 'window_signed_flow_usdc',
 'window_imbalance',
 'trailing_60s_trade_count',
 'trailing_60s_volume_usdc',
 'trailing_60s_signed_flow_usdc',
 'trailing_60s_imbalance',
 'taker_base_fee',
 'maker_base_fee',
 'market_date']

In [ ]:
# construct feature matrix, X, y
# note: volumes/counts are log transformed (skewed -> now proportional)
# signed flow is signed-log transformed 

LOG_COLS = [
    "price_second_trade_count", "price_second_volume_usdc",
    "window_trade_count", "window_volume_usdc",
    "trailing_60s_trade_count", "trailing_60s_volume_usdc",
]
SIGNED_LOG_COLS = ["window_signed_flow_usdc", "trailing_60s_signed_flow_usdc"]

df_model = df_eda_v1.with_columns(
    (pl.col("up_price") / (1 - pl.col("up_price"))).log().alias("up_logit"),
    *[pl.col(c).log1p().alias(f"log_{c}") for c in LOG_COLS],
    *[(pl.col(c).sign() * pl.col(c).abs().log1p()).alias(f"slog_{c}") for c in SIGNED_LOG_COLS],
)

feature_cols = [
    "up_logit",
    "price_age_seconds",
    "log_price_second_trade_count", "log_price_second_volume_usdc",
    "log_window_trade_count", "log_window_volume_usdc",
    "slog_window_signed_flow_usdc", "window_imbalance",
    "log_trailing_60s_trade_count", "log_trailing_60s_volume_usdc",
    "slog_trailing_60s_signed_flow_usdc", "trailing_60s_imbalance",
]

assert df_model.select(feature_cols).null_count().sum_horizontal().item() == 0
assert np.isfinite(df_model.select(feature_cols).to_numpy()).all()

In [ ]:
# features parquet -> metadata (ids / timing for splits), target, raw up_price (price baseline), features

META_COLS = ["market_id", "market_date", "window_start_ts", "prediction_ts", "end_ts", "up_price"]
TARGET_COL = "resolution"

df_features = df_model.select(META_COLS + [TARGET_COL] + feature_cols)

In [ ]:
PROCESSED_DATA_DIR = Path("../data/processed")
OUTPUT_PATH = PROCESSED_DATA_DIR / "timeseventeen_btc5m_features_v1.parquet"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_features.write_parquet(OUTPUT_PATH)

print(f"Saved {df_features.height:,} markets, {len(feature_cols)} features to {OUTPUT_PATH}")